# Lecture Pipeline: Whisper → Transcripts → Qwen → Notes

Unified EN/RU notebook for Google Colab with a **T4 GPU 16 GB** and Google Drive.
Run the four code cells top to bottom using **Run all**.

## What the pipeline does

1. Sets up folders and an isolated Whisper/pyannote environment.
2. Checks imports and GPU availability without loading large models.
3. Transcribes audio in batched mode and creates `Raw/*_raw.txt` and `Transcripts/*.txt`.
   Caching is fingerprint-based (file size + mtime): changed audio is re-transcribed, unchanged is skipped.
   A missing version is recovered from the existing one without new inference when possible.
4. If `GENERATE_NOTE=True`, it launches Qwen in a separate process, extracts metadata
   (guided by a JSON schema, with retries kept as a fallback), creates the Markdown note,
   and shuts down the model server.

## Language parameter

`NOTE_LANGUAGE` in the first cell controls the language of the LLM prompts and the generated notes:

- `"ru"` — Russian prompts, notes in Russian (default);
- `"en"` — English prompts, notes in English.

Metadata (title/topic/tags) is always extracted in English. Notebook interface text is always in English.

## Long lectures

If a transcript does not fit into the context, it is split automatically into parts:
a note is generated for the first part, then the model updates the note part by part.
No manual splitting required.

## Folders

- `lectures/Raw/` — clean transcription without timestamps and speakers.
- `lectures/Transcripts/` — transcription with optional timestamps and speakers.
- `lectures/Notes/` — notes. The nested `Notes/.metadata/` stores validated metadata.
- `lectures/Cache/ASR/` — transcription manifests (audio fingerprint + settings).
- `/content/qwen3-14b-gguf/` — Qwen GGUF file (~9 GB) on the local Colab disk. It is deleted and
  re-downloaded after a runtime reset.

Model: [Official Qwen3-14B Q4_K_M GGUF weights](https://huggingface.co/Qwen/Qwen3-14B-GGUF/blob/main/Qwen3-14B-Q4_K_M.gguf),
running via [llama.cpp](https://github.com/ggml-org/llama.cpp/releases) with CUDA 12.8.
Context is set to 32K; if T4 runs out of memory during loading, the code tries 24K, then 16K.
The KV cache of the shared prompt prefix is reused (`--cache-reuse`).

**First run:** downloading the Qwen model takes about 9 GB and depends on network speed.

**Languages:** `LANGUAGE` ("auto" by default) is the transcription language for Whisper;
`ASR_BATCH_SIZE` is the batched transcription batch.

**Important:** before running speaker diarization, accept the access terms for
[pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1)
and add `HF_TOKEN` to Colab Secrets with Notebook access enabled. All settings are in the first cell.


In [ ]:
# STAGE 1. SETTINGS AND ENVIRONMENT PREPARATION
# This block can be run repeatedly: it doesn't reinstall the environment unnecessarily.

import os
import re
import sys
import json
import time
import shutil
import tarfile
import urllib.request
import subprocess
import pty
import codecs
from pathlib import Path

from google.colab import drive, userdata

# ============================================================
# SETTINGS — change values only in this section
# ============================================================

ROOT_DIR = Path("/content/drive/MyDrive/lectures")
RAW_DIR = ROOT_DIR / "Raw"
TRANSCRIPTS_DIR = ROOT_DIR / "Transcripts"
NOTES_DIR = ROOT_DIR / "Notes"
METADATA_DIR = NOTES_DIR / ".metadata"

# Qwen weights are stored on the local disk of the current Colab machine, not in Google Drive.
# Important: after a runtime reset, local files are deleted, so the model will have to be downloaded again.
MODEL_DIR = Path("/content/qwen3-14b-gguf")
MODEL_REPO = "Qwen/Qwen3-14B-GGUF"
MODEL_FILENAME = "Qwen3-14B-Q4_K_M.gguf"

MODEL_NAME = "large-v3"
# "auto" detects the language separately for each audio file; you can explicitly specify "ru" or "en".
LANGUAGE = "auto"

SAVE_RAW = True
ADD_TIMESTAMPS = False
ADD_SPEAKERS = False
GENERATE_NOTE = True
AI_INPUT_RAW = True

MAX_METADATA_ATTEMPTS = 5
SKIP_EXISTING = True
AI_CONTEXT_SIZE = 32768
AI_MAX_NOTE_TOKENS = 6000

# Unified language of the LLM prompts and the generated notes: "ru" or "en".
# Metadata (title/topic/tags) is extracted in English in both modes.
NOTE_LANGUAGE = "ru"

USE_BATCHED_ASR = True      # batched Whisper inference (faster-whisper >= 1.x)
ASR_BATCH_SIZE = 8

CACHE_DIR = ROOT_DIR / "Cache"
ASR_CACHE_DIR = CACHE_DIR / "ASR"

AUDIO_EXTENSIONS = {
    ".mp3", ".ogg", ".wav", ".flac", ".m4a", ".opus",
    ".aac", ".wma", ".mp4", ".mkv", ".webm"
}

# ============================================================
# Google Drive and Hugging Face secret
# ============================================================

drive.mount("/content/drive")

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if ADD_SPEAKERS and not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is required for speaker diarization. Add HF_TOKEN to Colab Secrets, "
        "enable Notebook access, and accept the terms for pyannote/speaker-diarization-community-1."
    )

for folder in (ROOT_DIR, RAW_DIR, TRANSCRIPTS_DIR, NOTES_DIR, METADATA_DIR, CACHE_DIR, ASR_CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if GENERATE_NOTE:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

if GENERATE_NOTE and AI_INPUT_RAW and not SAVE_RAW:
    raise ValueError("AI_INPUT_RAW=True requires SAVE_RAW=True. Enable SAVE_RAW or set AI_INPUT_RAW=False.")

print("Settings loaded.")
print(f"Audio and results: {ROOT_DIR}")
print(f"Raw: {RAW_DIR}")
print(f"Transcripts: {TRANSCRIPTS_DIR}")
print(f"Notes: {NOTES_DIR}")
print(f"AI note generation: {GENERATE_NOTE}")
if GENERATE_NOTE:
    print(f"Local Qwen weights folder: {MODEL_DIR} (does not take up space on Google Drive)")

# ------------------------------------------------------------------
# Language packs. LLM prompts target the note language; the notebook
# interface itself is always in English.
# ------------------------------------------------------------------
LANGUAGE_PACK = {
    "ru": {
        "system": r'''Ты — редактор учебных конспектов и аналитик содержания лекций.
Твоя задача — превратить автоматическую расшифровку в точную, содержательную и удобную для повторного изучения заметку.

Основные правила:
1. Сохраняй смысл и фактическое содержание источника. Не добавляй утверждения, определения, примеры или факты, которых нет в расшифровке.
2. Расшифровка может содержать русский, английский или смешанный русско-английский текст. Не считай английские фразы ошибками распознавания. Корректно понимай переключение языков.
3. Сохраняй оригинальное написание технических терминов, названий технологий, функций, библиотек, команд, формул и имён собственных.
4. Исправляй только очевидные ошибки распознавания, если правильный вариант однозначен из контекста. Не угадывай неясные имена, числа, формулы и термины.
5. Убирай речевой мусор и бессмысленные повторы, но сохраняй существенные уточнения, ограничения, исключения и оговорки.
6. Сохраняй определения, алгоритмы, формулы, команды, конкретные примеры, причинно-следственные связи и объяснения того, почему что-либо работает.
7. Не превращай заметку в стенограмму. Подробно перескажи смысл и сохрани все важные детали.
8. Не выдумывай дату лекции, домашнее задание, предмет или имя преподавателя. Если сведения неизвестны, укажи это явно.
9. Если источник противоречив или не позволяет уверенно восстановить смысл, обозначь неопределённость, а не придумывай объяснение.
10. Финальная заметка должна быть написана на русском языке естественно и грамотно, даже если исходный текст содержит английские фрагменты. Сохраняй оригинальное написание технических терминов, названий технологий, функций, библиотек, команд и имён собственных.
11. Верни только запрошенный результат, без вводных фраз и описания своих действий.''',
        "metadata_prompt": r'''Проанализируй расшифровку и извлеки метаданные учебной заметки.
Верни только один корректный JSON-объект. Не используй Markdown, тройные обратные кавычки или пояснения до и после JSON.

Формат:
{
  "title": "короткое информативное название на английском",
  "topic": "конкретная тема на английском",
  "subject": "учебный предмет или область на английском либо null",
  "lecture_date": null,
  "tags": ["tag-one", "tag-two", "tag-three"]
}

Правила:
- Название и тема должны точно отражать содержание, а не быть общими.
- Расшифровка может быть русской, английской или смешанной. Понимай оба языка, но возвращай значения title/topic/subject и теги на английском.
- lecture_date заполняй в формате ГГГГ-ММ-ДД только если дата самой лекции прямо указана или однозначно восстанавливается. Иначе null. Не используй текущую дату вместо даты лекции.
- tags: от 3 до 7 коротких тегов в lowercase kebab-case. Допускаются латиница, цифры, дефисы, а также символы «+» и «#» (например c++, c#).
- Не добавляй темы, которые в лекции не обсуждались.
- Верни все пять ключей. Неизвестный subject допускается как null.

Расшифровка:
{{TRANSCRIPT}}''',
        "metadata_fix_hint": "Предыдущий ответ не прошёл проверку. Исправь только перечисленные проблемы и верни полный JSON заново.\nПроблемы: {error}\nПредыдущий некорректный ответ:\n{response}",
        "note_header": "Создай итоговую заметку по материалу ниже.\n\nВАЖНО:\n- Вся готовая заметка, включая заголовки и содержимое, должна быть на русском языке.\n- Не включай YAML frontmatter: программа добавит проверенные метаданные сама.\n- Не выдумывай дату и домашние задания. Если домашнее задание не упоминалось, напиши: «Домашнее задание явно не упоминалось».\n- Подробно сохраняй полезную информацию, определения, шаги, формулы, команды, примеры и ограничения.\n- Раздел «Подробный конспект» должен быть содержательным конспектом по ходу лекции, а не дословной стенограммой.\n- Правильно обрабатывай английские фрагменты исходника. Переводи обычный текст на русский, но сохраняй оригинальное написание технических терминов, названий технологий, функций, библиотек и команд.\n- Верни только Markdown-разделы заметки, без вступления и тройных обратных кавычек.",
        "metadata_label": "Проверенные метаданные:",
        "template_label": "Русскоязычный шаблон и назначение разделов:",
        "transcript_label": "Расшифровка лекции:",
        "note_template": r'''# Коротко об лекции
В 1–3 коротких абзацах объясни тему, цель занятия и главную мысль.

# Быстрые детали
Собери определения, факты, формулы, команды, названия, значения и другие сведения, которые нужно быстро найти. Не дублируй большие фрагменты текста.

# Ключевые моменты
Выдели главные идеи, аргументы, связи между концепциями и причинно-следственные отношения. Организуй материал в логичном порядке.

# Полный лог
Подробный, но отредактированный конспект по ходу лекции. Сохрани объяснения, последовательности действий, алгоритмы, примеры, ограничения и важные детали. Это не дословная стенограмма и не краткое резюме.

# Домашнее задание
Перечисли только задания, упражнения, сроки и требования, которые явно были упомянуты. Если задания не было, честно укажи это.

# Итоги
Кратко собери выводы и знания, которые стоит запомнить после лекции. Не повторяй полностью раздел «Коротко об лекции».''',
        "part_note_hint": "(Это часть {0} из {1} всей расшифровки. Метаданные описывают всю лекцию, но пока обработана только эта часть; заметка будет достроена следующими частями.)",
        "incremental_prompt": '''Ниже приведена заметка по началу лекции и следующая часть расшифровки (часть {{N}} из {{TOTAL}}).
Переработай заметку, вписав содержание новой части в существующие разделы.

Разделы заметки строго такие: {{HEADINGS}}.
Правила:
- Сохраняй все важные сведения из уже написанной заметки и добавляй новые факты из этой части.
- Не выдумывай факты. Если в части упоминается домашнее задание — занеси его в соответствующий раздел.
- Раздел «{{DETAILED}}» дополняется: продолжай его в логичном порядке по ходу лекции.
- Верни полную обновлённую заметку во всех разделах, без YAML frontmatter, без вступления и тройных обратных кавычек.

Текущая заметка:
{{NOTE_SO_FAR}}

Часть {{N}} из {{TOTAL}} расшифровки:
{{PART}}''',
        "sections": {
            "Коротко о лекции": ["Коротко об лекции", "Summary"],
            "Быстрые детали": ["Quick Details"],
            "Ключевые моменты": ["Key Points"],
            "Подробный конспект": ["Полный лог", "Detailed Notes"],
            "Домашнее задание": ["Homework"],
            "Итоги": ["Takeaways"],
        },
        "detailed_heading": "Подробный конспект",
        "heading_fmt": "«{}»",
        "fallback_homework": "Домашнее задание явно не упоминалось в исходной лекции.",
        "fallback_section": "Раздел не удалось сформировать по исходному материалу.",
        "frontmatter": {"date": "Дата лекции", "date_unspecified": "Не указана", "topic": "Тема"},
    },
    "en": {
        "system": r'''You are an educational notes editor and lecture content analyst.
Your task is to turn an automatic transcription into an accurate, comprehensive, and easy-to-review note.

Main rules:
1. Preserve the meaning and factual content of the source. Do not add statements, definitions, examples, or facts that are not in the transcription.
2. The transcription may contain Russian, English, or mixed Russian-English text. Do not treat English phrases as recognition errors. Correctly understand language switching.
3. Preserve the original spelling of technical terms, technology names, functions, libraries, commands, formulas, and proper names.
4. Correct only obvious recognition errors if the correct option is unambiguous from the context. Do not guess unclear names, numbers, formulas, and terms.
5. Remove speech disfluencies and meaningless repetitions, but preserve essential clarifications, limitations, exceptions, and caveats.
6. Preserve definitions, algorithms, formulas, commands, specific examples, cause-and-effect relationships, and explanations of why something works.
7. Do not turn the note into a verbatim transcript. Summarize the meaning in detail and keep all important details.
8. Do not invent the lecture date, homework, subject, or instructor name. If the information is unknown, state this explicitly.
9. If the source is contradictory or does not allow you to confidently reconstruct the meaning, indicate uncertainty rather than inventing an explanation.
10. The final note must be written in English naturally and grammatically, even if the source text contains Russian fragments. Preserve the original spelling of technical terms, technology names, functions, libraries, commands, and proper names.
11. Return only the requested result, without introductory phrases or descriptions of your actions.''',
        "metadata_prompt": r'''Analyze the transcription and extract metadata for the educational note.
Return only one valid JSON object. Do not use Markdown, triple backticks, or explanations before and after the JSON.

Format:
{
  "title": "short informative title in English",
  "topic": "specific topic in English",
  "subject": "academic subject or field in English or null",
  "lecture_date": null,
  "tags": ["tag-one", "tag-two", "tag-three"]
}

Rules:
- The title and topic must accurately reflect the content, not be generic.
- The transcription may be in Russian, English, or mixed. Understand both languages, but return title/topic/subject values and tags in English.
- Fill in lecture_date in YYYY-MM-DD format only if the date of the lecture itself is explicitly stated or unambiguously recoverable. Otherwise null. Do not use the current date instead of the lecture date.
- tags: 3 to 7 short relevant tags, lowercase, kebab-case, in English. Latin letters, digits, hyphens are allowed, as well as "+" and "#" (e.g. c++, c#).
- Do not add topics that were not discussed in the lecture.
- Return all five keys. Unknown subject is allowed as null.

Transcription:
{{TRANSCRIPT}}''',
        "metadata_fix_hint": "The previous response failed validation. Fix only the listed issues and return the full JSON again.\nIssues: {error}\nPrevious invalid response:\n{response}",
        "note_header": "Create a final note based on the material below.\n\nIMPORTANT:\n- The entire finished note, including headings and content, must be in English.\n- Do not include YAML frontmatter: the program will add validated metadata itself.\n- Do not invent dates and homework. If homework was not mentioned, write: \"Homework was not explicitly mentioned\".\n- Preserve useful information, definitions, steps, formulas, commands, examples, and limitations in detail.\n- The \"Detailed Notes\" section should be a comprehensive summary of the lecture, not a verbatim transcript.\n- Correctly handle Russian fragments of the source. Translate regular text to English, but preserve the original spelling of technical terms, technology names, functions, libraries, and commands.\n- Return only the Markdown sections of the note, without an introduction and triple backticks.",
        "metadata_label": "Validated metadata:",
        "template_label": "English template and section purposes:",
        "transcript_label": "Lecture transcription:",
        "note_template": r'''# About the Lecture
In 1–3 short paragraphs, explain the topic, the goal of the session, and the main idea.

# Quick Details
Collect definitions, facts, formulas, commands, names, values, and other information that needs to be found quickly. Do not duplicate large chunks of text.

# Key Points
Highlight the main ideas, arguments, connections between concepts, and cause-and-effect relationships. Organize the material in a logical order.

# Detailed Notes
A detailed but edited summary of the lecture. Preserve explanations, sequences of actions, algorithms, examples, limitations, and important details. This is not a verbatim transcript or a brief summary.

# Homework
List only the assignments, exercises, deadlines, and requirements that were explicitly mentioned. If there was no homework, honestly state this.

# Takeaways
Briefly summarize the conclusions and knowledge worth remembering after the lecture. Do not completely repeat the "About the Lecture" section.''',
        "part_note_hint": "(This is part {0} of {1} of the whole transcript. The metadata describes the entire lecture, but only this part has been processed so far; the note will be extended by the following parts.)",
        "incremental_prompt": '''Below is the note built from the beginning of the lecture and the next part of the transcript (part {{N}} of {{TOTAL}}).
Rework the note to incorporate the content of the new part into the existing sections.

The note sections are strictly: {{HEADINGS}}.
Rules:
- Keep every important fact already present in the note; add the new facts from this part.
- Do not invent facts. If this part mentions homework, put it into the Homework section.
- The "{{DETAILED}}" section grows: continue it in a logical order following the lecture.
- Return the complete updated note with all sections, without YAML frontmatter, without a preamble and without triple backticks.

Current note:
{{NOTE_SO_FAR}}

Transcript part {{N}} of {{TOTAL}}:
{{PART}}''',
        "sections": {
            "About the Lecture": ["Коротко об лекции", "Коротко о лекции", "Summary"],
            "Quick Details": ["Быстрые детали"],
            "Key Points": ["Ключевые моменты"],
            "Detailed Notes": ["Полный лог", "Подробный конспект"],
            "Homework": ["Домашнее задание"],
            "Takeaways": ["Итоги"],
        },
        "detailed_heading": "Detailed Notes",
        "heading_fmt": "\"{}\"",
        "fallback_homework": "Homework was not explicitly mentioned in the source lecture.",
        "fallback_section": "The section could not be formed from the source material.",
        "frontmatter": {"date": "Lecture date", "date_unspecified": "Not specified", "topic": "Topic"},
    },
}

if NOTE_LANGUAGE not in LANGUAGE_PACK:
    raise ValueError(f"NOTE_LANGUAGE must be one of {sorted(LANGUAGE_PACK)}, got {NOTE_LANGUAGE!r}")
PACK = LANGUAGE_PACK[NOTE_LANGUAGE]
_note_language_label = "Russian" if NOTE_LANGUAGE == "ru" else "English"
print(f"Note language: {NOTE_LANGUAGE} - prompts and notes in {_note_language_label}; metadata always in English")

# ============================================================
# Helper functions for live output of the child Python process
# ============================================================

def run_streaming(command, env, cwd=None):
    """Runs a process via PTY: print and tqdm are visible in Colab immediately."""
    master_fd, slave_fd = pty.openpty()
    process = subprocess.Popen(
        command,
        stdin=slave_fd,
        stdout=slave_fd,
        stderr=slave_fd,
        env=env,
        cwd=cwd,
        close_fds=True,
    )
    os.close(slave_fd)
    decoder = codecs.getincrementaldecoder("utf-8")("replace")

    try:
        while True:
            try:
                data = os.read(master_fd, 4096)
                if not data:
                    break
                sys.stdout.write(decoder.decode(data))
                sys.stdout.flush()
            except OSError:
                # Linux PTY usually returns EIO when the child process has exited.
                break
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        try:
            sys.stdout.write(decoder.decode(b"", final=True))
            sys.stdout.flush()
        except Exception:
            pass
        os.close(master_fd)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Child process exited with code {return_code}: {command[0]}")
    return return_code

# ============================================================
# FFmpeg
# ============================================================

if shutil.which("ffmpeg") is None:
    print("Installing FFmpeg...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
else:
    print("FFmpeg found.")

# ============================================================
# Separate Python environment for Whisper and pyannote
# ============================================================

VENV = "/content/whisper_env"
PYTHON = f"{VENV}/bin/python"
READY_FILE = Path(VENV) / ".lecture_pipeline_ready"

if not Path(PYTHON).exists() or not READY_FILE.exists():
    if Path(VENV).exists():
        print("Removing incomplete/unknown Whisper environment...")
        shutil.rmtree(VENV)

    print("Installing uv to create an isolated environment...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    print("Creating Python 3.12 venv...")
    subprocess.run(["uv", "venv", "--python", "3.12", VENV], check=True)

    print("Installing PyTorch and TorchAudio with CUDA 12.8...")
    subprocess.run([
        "uv", "pip", "install", "--python", PYTHON,
        "--index-url", "https://download.pytorch.org/whl/cu128",
        "torch==2.11.0", "torchaudio==2.11.0"
    ], check=True)

    print("Installing Whisper and pyannote...")
    subprocess.run([
        "uv", "pip", "install", "--python", PYTHON,
        "pyannote.audio==4.0.7",
        "faster-whisper==1.2.1",
        "av<19",
        "tqdm",
        "huggingface_hub"
    ], check=True)

    READY_FILE.write_text("ready", encoding="utf-8")
    print("Whisper environment is ready.")
else:
    print("Whisper environment is already prepared — skipping reinstallation.")

# ============================================================
# Pre-built llama.cpp with CUDA 12.8 for local Qwen
# We only download the binaries; the Qwen weights will be downloaded in the 4th cell.
# ============================================================

LLAMA_DIR = Path("/content/llama_cpp")
LLAMA_DIR.mkdir(parents=True, exist_ok=True)


def valid_tarball(path):
    try:
        with tarfile.open(path, "r:gz") as archive:
            archive.getmembers()
        return True
    except Exception:
        return False


def download_with_progress(url, destination):
    destination = Path(destination)
    request = urllib.request.Request(url, headers={"User-Agent": "Colab-Lecture-Pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:
        total = int(response.headers.get("Content-Length", "0"))
        downloaded = 0
        with open(destination, "wb") as output:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded * 100 // total
                    print(f"\rDownloading binary: {pct}% ({downloaded // (1024 * 1024)} MiB)", end="", flush=True)
    print()


def find_llama_server():
    candidates = [p for p in LLAMA_DIR.rglob("llama-server") if p.is_file()]
    return candidates[0] if candidates else None


LLAMA_SERVER_PATH = find_llama_server()
LLAMA_READY = LLAMA_DIR / ".ready"

if GENERATE_NOTE and (LLAMA_SERVER_PATH is None or not LLAMA_READY.exists()):
    print("Looking for a pre-built llama.cpp with CUDA 12.8...")
    releases_url = "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=20"
    request = urllib.request.Request(releases_url, headers={"User-Agent": "Colab-Lecture-Pipeline", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(request, timeout=60) as response:
        releases = json.load(response)

    selected = None
    main_asset = None
    cudart_asset = None

    for release in releases:
        assets = {asset["name"]: asset["browser_download_url"] for asset in release.get("assets", [])}
        mains = [name for name in assets if name.startswith("llama-") and "bin-ubuntu-cuda-12.8-x64.tar.gz" in name]
        for main_name in mains:
            cudart_name = "cudart-" + main_name
            if cudart_name in assets:
                selected = release
                main_asset = (main_name, assets[main_name])
                cudart_asset = (cudart_name, assets[cudart_name])
                break
        if selected:
            break

    if not selected:
        raise RuntimeError("Could not find a published llama.cpp build with CUDA 12.8. Try re-running the cell later.")

    print(f"Selected llama.cpp build: {selected.get('tag_name', 'unknown')}")
    for name, url in (main_asset, cudart_asset):
        archive_path = Path("/content") / name
        if not archive_path.exists() or archive_path.stat().st_size < 1024 * 1024 or not valid_tarball(archive_path):
            if archive_path.exists():
                archive_path.unlink()
            print(f"Downloading {name}...")
            download_with_progress(url, archive_path)
        print(f"Extracting {name}...")
        with tarfile.open(archive_path, "r:gz") as archive:
            archive.extractall(LLAMA_DIR, filter="data")

    LLAMA_SERVER_PATH = find_llama_server()
    if LLAMA_SERVER_PATH is not None:
        LLAMA_READY.write_text("ready", encoding="utf-8")

if GENERATE_NOTE and LLAMA_SERVER_PATH is None:
    raise RuntimeError("Executable file llama-server not found after extraction.")

LLAMA_LD_LIBRARY_PATH = ""
if GENERATE_NOTE:
    for path in LLAMA_DIR.rglob("llama-server"):
        path.chmod(path.stat().st_mode | 0o111)
    library_dirs = {str(p.parent) for p in LLAMA_DIR.rglob("*.so*") if p.is_file()}
    library_dirs.update({"/usr/local/cuda/lib64", "/usr/local/nvidia/lib64"})
    old_ld = os.environ.get("LD_LIBRARY_PATH", "")
    LLAMA_LD_LIBRARY_PATH = ":".join(sorted(library_dirs)) + (":" + old_ld if old_ld else "")

    check_env = os.environ.copy()
    check_env["LD_LIBRARY_PATH"] = LLAMA_LD_LIBRARY_PATH
    check = subprocess.run([str(LLAMA_SERVER_PATH), "--version"], env=check_env, text=True, capture_output=True)
    if check.returncode != 0:
        print(check.stdout)
        print(check.stderr)
        raise RuntimeError("Failed to start llama-server. Check CUDA access and retry setup.")
    print(f"llama.cpp is ready: {LLAMA_SERVER_PATH}")
else:
    print("AI is disabled: llama.cpp binaries were not installed.")

print("\nStage 1 completed. Settings and paths are ready.")

## Stage 2 — Environment check

Imports, GPU and FFmpeg are verified. Large models are not loaded here.


In [ ]:
# STAGE 2. ENVIRONMENT CHECK
# We only check imports and GPU availability. Large models are NOT loaded here
# to avoid wasting time and VRAM before the actual transcription.

preflight_script = r'''
import os
import sys
import shutil
import warnings

warnings.filterwarnings("ignore", message="torchcodec is not installed correctly")

print("Python:", sys.version.split()[0], flush=True)

import numpy as np
import torch
from faster_whisper import WhisperModel

print("NumPy:", np.__version__, flush=True)
print("Torch:", torch.__version__, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)

if not torch.cuda.is_available():
    raise RuntimeError("GPU is unavailable. Select Runtime type → T4 GPU.")

if os.environ.get("ADD_SPEAKERS", "1") == "1":
    from pyannote.audio import Pipeline
    print("pyannote import: OK", flush=True)
else:
    print("Speaker diarization is disabled — pyannote is not imported.", flush=True)

if shutil.which("ffmpeg") is None:
    raise RuntimeError("FFmpeg not found.")

print("Imports and GPU: OK", flush=True)
'''

preflight_path = "/content/lecture_preflight.py"
Path(preflight_path).write_text(preflight_script, encoding="utf-8")

check_env = os.environ.copy()
check_env["MPLBACKEND"] = "Agg"
check_env["ADD_SPEAKERS"] = "1" if ADD_SPEAKERS else "0"
check_env["PYTHONUNBUFFERED"] = "1"

run_streaming([PYTHON, "-u", preflight_path], check_env)

if GENERATE_NOTE:
    check_env = os.environ.copy()
    check_env["LD_LIBRARY_PATH"] = LLAMA_LD_LIBRARY_PATH
    check = subprocess.run([str(LLAMA_SERVER_PATH), "--version"], env=check_env, text=True, capture_output=True)
    print("llama.cpp:", (check.stdout or check.stderr).strip())

print("Check completed. Whisper and Qwen weights have not been loaded yet.")

## Stage 3 — Transcription

Whisper transcribes in batched mode (automatic fallback to single-stream; it definitely keeps working with
pyannote speaker diarization and timestamps). Caching is fingerprint-based (size + mtime) with a per-file
manifest in `Cache/ASR/`: changed audio is re-transcribed, unchanged is skipped. Missing versions are
recovered from the existing ones without re-inference when possible.


In [ ]:
# STAGE 3. TRANSCRIPTION (batched Whisper, fingerprint-based cache)
# Whisper and pyannote are run in a separate process. When the process exits,
# VRAM is freed by the OS; the next stage uses a different process.

asr_script = r'''
import os
import re
import json
import time
import subprocess
import warnings
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm
from faster_whisper import WhisperModel, BatchedInferencePipeline

warnings.filterwarnings("ignore", message="torchcodec is not installed correctly")

INPUT_DIR = Path(os.environ["INPUT_DIR"])
RAW_DIR = Path(os.environ["RAW_DIR"])
TRANSCRIPTS_DIR = Path(os.environ["TRANSCRIPTS_DIR"])
ASR_CACHE_DIR = Path(os.environ["ASR_CACHE_DIR"])
MODEL_NAME = os.environ.get("MODEL_NAME", "large-v3")
LANGUAGE = os.environ.get("LANGUAGE", "ru")
if LANGUAGE.lower() == "auto":
    LANGUAGE = None
HF_TOKEN = os.environ.get("HF_TOKEN", "")
SAVE_RAW = os.environ.get("SAVE_RAW", "1") == "1"
ADD_TIMESTAMPS = os.environ.get("ADD_TIMESTAMPS", "1") == "1"
ADD_SPEAKERS = os.environ.get("ADD_SPEAKERS", "1") == "1"
USE_BATCHED = os.environ.get("USE_BATCHED_ASR", "1") == "1"
BATCH_SIZE = int(os.environ.get("ASR_BATCH_SIZE", "8"))

AUDIO_EXTENSIONS = {
    ".mp3", ".ogg", ".wav", ".flac", ".m4a", ".opus",
    ".aac", ".wma", ".mp4", ".mkv", ".webm"
}
OUTPUT_FOLDER_NAMES = {"Raw", "Transcripts", "Notes", "Cache"}

RAW_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
ASR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FLAGS = {"save_raw": SAVE_RAW, "timestamps": ADD_TIMESTAMPS, "speakers": ADD_SPEAKERS}


def output_stem(audio_path):
    relative = audio_path.relative_to(INPUT_DIR).with_suffix("")
    return "__".join(relative.parts)


def valid_file(path):
    return path.exists() and path.is_file() and path.stat().st_size > 0


def fingerprint(path):
    stat = path.stat()
    return {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


def load_manifest(manifest_path):
    if not manifest_path.exists():
        return None
    try:
        data = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        return None
    if not isinstance(data, dict) or "fingerprint" not in data:
        return None
    return data


def manifest_matches(saved, current_fingerprint):
    """A missing manifest is treated as a legacy cache: it is trusted for now
    and overwritten with a fresh one after this run."""
    if saved is None:
        return True
    return (
        saved.get("fingerprint") == current_fingerprint
        and saved.get("model") == MODEL_NAME
        and saved.get("language") == (LANGUAGE or "auto")
        and saved.get("flags") == FLAGS
    )


def write_manifest(manifest_path, current_fingerprint):
    data = {
        "fingerprint": current_fingerprint,
        "model": MODEL_NAME,
        "language": LANGUAGE or "auto",
        "flags": FLAGS,
    }
    temporary = manifest_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(manifest_path)


def fmt_time(seconds):
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def raw_from_transcript(text):
    lines = []
    for line in text.splitlines():
        line = re.sub(r"^\[\d{2}:\d{2}:\d{2}\s*-\s*\d{2}:\d{2}:\d{2}\]\s*", "", line.strip())
        line = re.sub(r"^Speaker\s+(?:\d+|\?)\s*:\s*", "", line)
        if line:
            lines.append(line)
    return "\n".join(lines).strip()


def load_audio_for_pyannote(audio_path):
    print("    Decoding audio for pyannote via FFmpeg...", flush=True)
    process = subprocess.run(
        ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", str(audio_path),
         "-f", "f32le", "-ac", "1", "-ar", "16000", "pipe:1"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if process.returncode != 0:
        error = process.stderr.decode("utf-8", errors="replace")
        raise RuntimeError(f"FFmpeg failed to read audio: {error}")
    audio = np.frombuffer(process.stdout, dtype=np.float32).copy()
    if audio.size == 0:
        raise RuntimeError("FFmpeg returned empty audio.")
    print("    Audio is ready.", flush=True)
    return {"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": 16000}


def assign_speakers(segments, diarization):
    output = []
    for seg in segments:
        best_speaker = None
        best_overlap = 0.0
        for d_start, d_end, speaker in diarization:
            overlap = min(seg.end, d_end) - max(seg.start, d_start)
            if overlap > best_overlap:
                best_speaker = speaker
                best_overlap = overlap
        output.append((seg.start, seg.end, best_speaker, seg.text.strip()))
    return output


def build_transcript(segments, diarization):
    mapping = {}
    for _, _, speaker in diarization:
        if speaker not in mapping:
            mapping[speaker] = f"Speaker {len(mapping) + 1}"

    assigned = assign_speakers(segments, diarization) if ADD_SPEAKERS else [
        (seg.start, seg.end, None, seg.text.strip()) for seg in segments
    ]
    lines = []
    for start, end, speaker, text in assigned:
        if not text:
            continue
        parts = []
        if ADD_TIMESTAMPS:
            parts.append(f"[{fmt_time(start)} - {fmt_time(end)}]")
        if ADD_SPEAKERS:
            parts.append(mapping.get(speaker, "Speaker ?") + ":")
        prefix = " ".join(parts)
        lines.append(f"{prefix} {text}" if prefix else text)
    return "\n".join(lines).strip()


def transcribe(audio_model, batch_pipeline, path):
    common = {"language": LANGUAGE, "vad_filter": True, "word_timestamps": True}
    if batch_pipeline is not None:
        try:
            return batch_pipeline.transcribe(str(path), batch_size=BATCH_SIZE, beam_size=5, **common)
        except TypeError:
            # An older pipeline build may not accept some options; fall back to single-stream.
            return audio_model.transcribe(str(path), beam_size=5, **common)
    return audio_model.transcribe(str(path), beam_size=5, **common)


files = []
for path in INPUT_DIR.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
        continue
    relative = path.relative_to(INPUT_DIR)
    if any(part in OUTPUT_FOLDER_NAMES for part in relative.parts[:-1]):
        continue
    files.append(path)
files.sort()

print(f"Found audio files: {len(files)}", flush=True)
for path in files:
    print(" -", path.name, flush=True)

if not files:
    print("No audio to process.", flush=True)
    raise SystemExit(0)

model = None
batch_pipeline = None
diar_pipe = None

completed = 0
failed = 0
progress = tqdm(files, desc="Transcription", unit="file", dynamic_ncols=True, mininterval=1.0)

for index, audio_path in enumerate(progress, 1):
    stem = output_stem(audio_path)
    raw_path = RAW_DIR / f"{stem}_raw.txt"
    transcript_path = TRANSCRIPTS_DIR / f"{stem}.txt"
    manifest_path = ASR_CACHE_DIR / f"{stem}.json"
    current_fp = fingerprint(audio_path)
    saved_manifest = load_manifest(manifest_path)
    cache_ok = manifest_matches(saved_manifest, current_fp)

    need_raw = SAVE_RAW and not valid_file(raw_path)
    need_transcript = not valid_file(transcript_path)

    if not cache_ok:
        print(f"\n[{index}/{len(files)}] Audio or settings changed; re-transcribing — {audio_path.name}", flush=True)
        need_raw = SAVE_RAW
        need_transcript = True
    elif saved_manifest is None:
        # Legacy cache without a manifest: keep the results, write a manifest for the next runs.
        write_manifest(manifest_path, current_fp)

    if not need_raw and not need_transcript:
        if cache_ok:
            print(f"\n[{index}/{len(files)}] Skipping: cache hit — {audio_path.name}", flush=True)
        completed += 1
        progress.set_postfix(done=completed, errors=failed)
        continue

    # Cross-recovery is only safe when the existing files match the current audio fingerprint.
    if cache_ok and need_raw and valid_file(transcript_path):
        raw_path.write_text(raw_from_transcript(transcript_path.read_text(encoding="utf-8")), encoding="utf-8")
        need_raw = not valid_file(raw_path)
        print(f"\n[{index}/{len(files)}] Raw recovered from existing transcript: {raw_path.name}", flush=True)

    if cache_ok and need_transcript and valid_file(raw_path) and not ADD_TIMESTAMPS and not ADD_SPEAKERS:
        transcript_path.write_text(raw_path.read_text(encoding="utf-8"), encoding="utf-8")
        need_transcript = not valid_file(transcript_path)
        print(f"\n[{index}/{len(files)}] Transcript created from Raw without re-inference.", flush=True)

    if not need_raw and not need_transcript:
        completed += 1
        progress.set_postfix(done=completed, errors=failed)
        continue

    t0 = time.time()
    print(f"\n[{index}/{len(files)}] File: {audio_path.name}", flush=True)

    try:
        if model is None:
            print("Loading Whisper large-v3...", flush=True)
            model = WhisperModel(MODEL_NAME, device="cuda", compute_type="float16")
            if USE_BATCHED:
                try:
                    batch_pipeline = BatchedInferencePipeline(model=model)
                    print(f"Batched mode is ON (batch size {BATCH_SIZE}).", flush=True)
                except Exception as error:
                    batch_pipeline = None
                    print(f"Batched mode is unavailable ({type(error).__name__}); single-stream mode is used.", flush=True)
            print("Whisper loaded.", flush=True)

        print("    [1/3] Speech recognition...", flush=True)
        segments_iter, _ = transcribe(model, batch_pipeline, audio_path)
        segments = list(segments_iter)
        print(f"    [1/3] Done: {len(segments)} segments.", flush=True)

        if need_raw:
            raw_text = "\n".join(seg.text.strip() for seg in segments if seg.text.strip()).strip()
            raw_path.write_text(raw_text, encoding="utf-8")
            print(f"    Raw saved: {raw_path}", flush=True)

        diarization = []
        if ADD_SPEAKERS and need_transcript:
            if not HF_TOKEN:
                raise RuntimeError("ADD_SPEAKERS is enabled, but HF_TOKEN is empty.")
            if diar_pipe is None:
                print("Loading pyannote community-1...", flush=True)
                from pyannote.audio import Pipeline
                diar_pipe = Pipeline.from_pretrained(
                    "pyannote/speaker-diarization-community-1",
                    token=HF_TOKEN,
                )
                if diar_pipe is None:
                    raise RuntimeError("Failed to load pyannote pipeline.")
                diar_pipe.to(torch.device("cuda"))
                print("Pyannote loaded.", flush=True)
            waveform = load_audio_for_pyannote(audio_path)
            print("    [2/3] Speaker diarization...", flush=True)
            result = diar_pipe(waveform)
            diarization = [
                (turn.start, turn.end, speaker)
                for turn, _, speaker in result.speaker_diarization.itertracks(yield_label=True)
            ]
            print(f"    [2/3] Done: {len(diarization)} speaker intervals.", flush=True)
        elif ADD_SPEAKERS and not need_transcript:
            print("    Diarization not needed: transcript already exists.", flush=True)

        if need_transcript:
            print("    [3/3] Building transcript...", flush=True)
            transcript_text = build_transcript(segments, diarization)
            transcript_path.write_text(transcript_text, encoding="utf-8")
            print(f"    Transcript saved: {transcript_path}", flush=True)

        write_manifest(manifest_path, current_fp)
        completed += 1
        print(f"Done in {time.time() - t0:.1f} sec.", flush=True)
        progress.set_postfix(done=completed, errors=failed)

    except Exception as error:
        failed += 1
        print(f"ERROR {type(error).__name__}: {error}", flush=True)
        progress.set_postfix(done=completed, errors=failed)

progress.close()
print(f"\nTranscription completed. Successful: {completed}, errors: {failed}.", flush=True)
print(f"Raw: {RAW_DIR}", flush=True)
print(f"Transcripts: {TRANSCRIPTS_DIR}", flush=True)
'''

asr_script_path = "/content/lecture_transcribe.py"
Path(asr_script_path).write_text(asr_script, encoding="utf-8")

asr_env = os.environ.copy()
asr_env.update({
    "INPUT_DIR": str(ROOT_DIR),
    "RAW_DIR": str(RAW_DIR),
    "TRANSCRIPTS_DIR": str(TRANSCRIPTS_DIR),
    "ASR_CACHE_DIR": str(ASR_CACHE_DIR),
    "MODEL_NAME": MODEL_NAME,
    "LANGUAGE": LANGUAGE,
    "HF_TOKEN": HF_TOKEN or "",
    "SAVE_RAW": "1" if SAVE_RAW else "0",
    "ADD_TIMESTAMPS": "1" if ADD_TIMESTAMPS else "0",
    "ADD_SPEAKERS": "1" if ADD_SPEAKERS else "0",
    "USE_BATCHED_ASR": "1" if USE_BATCHED_ASR else "0",
    "ASR_BATCH_SIZE": str(ASR_BATCH_SIZE),
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
})

run_streaming([PYTHON, "-u", asr_script_path], asr_env)
print("Transcription stage completed. Child Python process exited — its CUDA context is freed.")


## Stage 4 — AI note generation

Qwen produces the note; metadata is extracted with a JSON schema (`response_format`) and retries kept as a
fallback. Long transcripts are split automatically: the note is generated from the first part and then
updated part by part. The KV cache of the shared prompt prefix is reused (`--cache-reuse`).


In [ ]:
# STAGE 4. AI NOTES
# If GENERATE_NOTE=False, this stage will finish without loading Qwen weights.
# Qwen is run as a separate llama-server; after processing, it is guaranteed to stop.

if not GENERATE_NOTE:
    print("AI note generation is disabled (GENERATE_NOTE=False). Stage skipped.")
else:
    ai_template = r"""
import os
import re
import json
import time
import urllib.request
import urllib.error
import subprocess
import socket
from datetime import date
from pathlib import Path

from huggingface_hub import hf_hub_download

ROOT_DIR = Path(os.environ["INPUT_DIR"])
RAW_DIR = Path(os.environ["RAW_DIR"])
TRANSCRIPTS_DIR = Path(os.environ["TRANSCRIPTS_DIR"])
NOTES_DIR = Path(os.environ["NOTES_DIR"])
METADATA_DIR = Path(os.environ["METADATA_DIR"])
MODEL_DIR = Path(os.environ["MODEL_DIR"])
MODEL_REPO = os.environ["MODEL_REPO"]
MODEL_FILENAME = os.environ["MODEL_FILENAME"]
LLAMA_SERVER = os.environ["LLAMA_SERVER_PATH"]
LLAMA_LD_LIBRARY_PATH = os.environ["LLAMA_LD_LIBRARY_PATH"]
AI_INPUT_RAW = os.environ.get("AI_INPUT_RAW", "1") == "1"
AI_CONTEXT_SIZE = int(os.environ.get("AI_CONTEXT_SIZE", "32768"))
MAX_NOTE_TOKENS = int(os.environ.get("AI_MAX_NOTE_TOKENS", "6000"))
MAX_METADATA_ATTEMPTS = int(os.environ.get("MAX_METADATA_ATTEMPTS", "5"))

AUDIO_EXTENSIONS = {
    ".mp3", ".ogg", ".wav", ".flac", ".m4a", ".opus",
    ".aac", ".wma", ".mp4", ".mkv", ".webm"
}
OUTPUT_FOLDER_NAMES = {"Raw", "Transcripts", "Notes", "Cache"}

MODEL_DIR.mkdir(parents=True, exist_ok=True)

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


PORT = find_free_port()
BASE_URL = f"http://127.0.0.1:{PORT}"

print(f"Selected free port for llama-server: {PORT}", flush=True)

NOTES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = r'''%%SYSTEM_PROMPT%%'''
METADATA_PROMPT = r'''%%METADATA_PROMPT%%'''
METADATA_FIX_HINT = '''%%METADATA_FIX_HINT%%'''
NOTE_PROMPT_HEADER = '''%%NOTE_PROMPT_HEADER%%'''
METADATA_LABEL = '''%%METADATA_LABEL%%'''
TEMPLATE_LABEL = '''%%TEMPLATE_LABEL%%'''
TRANSCRIPT_LABEL = '''%%TRANSCRIPT_LABEL%%'''
NOTE_TEMPLATE = r'''%%NOTE_TEMPLATE%%'''
PART_NOTE_HINT = '''%%PART_NOTE_HINT%%'''
INCREMENTAL_PROMPT = r'''%%INCREMENTAL_PROMPT%%'''
SECTION_ALIASES = %%SECTIONS_JSON%%
FALLBACK_HOMEWORK = '''%%FALLBACK_HOMEWORK%%'''
FALLBACK_SECTION = '''%%FALLBACK_SECTION%%'''
FM_LABELS = %%FM_LABELS_JSON%%
DETAILED_HEADING = '''%%DETAILED%%'''
HOMEWORK_NAME = "%%HOMEWORK%%"
HEADING_FMT = %%HEADING_FMT_RAW%%


# Handles an optional reasoning block in the model answer.
_THINK_OPEN = "<" + "think" + ">"
_THINK_CLOSE = "</" + "think" + ">"

def strip_reasoning(text):
    return re.sub(_THINK_OPEN + r".*?" + _THINK_CLOSE, "", text, flags=re.DOTALL).strip()


# JSON schema guiding the metadata call; validation retries stay as a fallback.
RESPONSE_FORMAT_ENABLED = True
METADATA_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "topic": {"type": "string"},
        "subject": {"type": ["string", "null"]},
        "lecture_date": {"type": ["string", "null"]},
        "tags": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["title", "topic", "subject", "lecture_date", "tags"],
}


def post_json(path, payload, timeout=3600):
    data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(
        BASE_URL + path,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def get_json(path, timeout=10):
    with urllib.request.urlopen(BASE_URL + path, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def stream_completion(body):
    '''Sends a chat completion request and collects the streamed answer.'''
    request = urllib.request.Request(
        BASE_URL + "/v1/chat/completions",
        data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    chunks = []
    pieces = 0
    finish_reason = None
    with urllib.request.urlopen(request, timeout=3600) as response:
        for raw_line in response:
            line = raw_line.decode("utf-8", errors="replace").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            try:
                event = json.loads(data)
            except json.JSONDecodeError:
                continue
            choices = event.get("choices", [])
            if not choices:
                continue
            choice = choices[0]
            delta = choice.get("delta", {})
            content = delta.get("content")
            if isinstance(content, str) and content:
                chunks.append(content)
                pieces += 1
                if pieces % 100 == 0:
                    print(f"  Generation continues: received {pieces} chunks...", flush=True)
            if choice.get("finish_reason") is not None:
                finish_reason = choice["finish_reason"]
    return "".join(chunks), finish_reason


def model_completion(system_prompt, user_prompt, max_tokens, temperature=0.1, json_schema=None):
    # Tokenize the content via the same server; leave a margin for chat formatting and response.
    combined = system_prompt + "\n\n" + user_prompt
    try:
        token_result = post_json("/tokenize", {"content": combined}, timeout=300)
        prompt_tokens = len(token_result.get("tokens", []))
    except Exception as error:
        # Conservative estimate — only a fallback if the tokenizer endpoint is unavailable.
        prompt_tokens = max(1, int(len(combined) / 1.5))
        print(f"  Warning: exact token count failed ({error}); used estimate {prompt_tokens}.", flush=True)

    available = CONTEXT_SIZE - prompt_tokens - 512
    if available < 256:
        raise RuntimeError(
            f"Input is too long for context {CONTEXT_SIZE}: about {prompt_tokens} tokens before generation."
        )
    output_limit = min(max_tokens, available)
    print(f"  Context: about {prompt_tokens} input tokens; response limit {output_limit}.", flush=True)

    body = {
        "model": "Qwen3-14B-Q4_K_M",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "top_p": 0.9,
        "max_tokens": output_limit,
        "stream": True,
    }
    if json_schema is not None:
        body["response_format"] = {"type": "json_object", "schema": json_schema}

    try:
        text, finish_reason = stream_completion(body)
    except urllib.error.HTTPError as error:
        # Older llama.cpp builds may not support response_format; retry once without it.
        if json_schema is None or error.code != 400:
            raise
        print("  The server rejected response_format; falling back to plain generation.", flush=True)
        body.pop("response_format", None)
        text, finish_reason = stream_completion(body)

    text = strip_reasoning(text).strip()
    if not text:
        raise RuntimeError("The model returned an empty response.")
    if finish_reason == "length":
        raise RuntimeError("The model's response was truncated by the token limit; the note was not saved as complete.")
    return text


# "+" and "#" are allowed so that technical tags like "c++" or "c#" pass validation.
TAG_PATTERN = r"[a-zа-яё0-9+#]+(?:-[a-zа-яё0-9+#]+)*"


def parse_metadata(text):
    cleaned = strip_reasoning(text)
    decoder = json.JSONDecoder()
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object in the response.")
    data, _ = decoder.raw_decode(cleaned[start:])
    if not isinstance(data, dict):
        raise ValueError("JSON must be an object.")

    required = {"title", "topic", "subject", "lecture_date", "tags"}
    missing = required - data.keys()
    if missing:
        raise ValueError(f"Missing keys: {sorted(missing)}")
    for key in ("title", "topic"):
        if not isinstance(data[key], str) or not data[key].strip():
            raise ValueError(f"{key} must be a non-empty string.")
    if data["subject"] is not None and not isinstance(data["subject"], str):
        raise ValueError("subject must be a string or null.")
    if data["lecture_date"] is not None:
        if not isinstance(data["lecture_date"], str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", data["lecture_date"]):
            raise ValueError("lecture_date must be a YYYY-MM-DD date or null.")
        from datetime import datetime
        try:
            datetime.strptime(data["lecture_date"], "%Y-%m-%d")
        except ValueError:
            raise ValueError("lecture_date contains an impossible date.")
    tags = data["tags"]
    if not isinstance(tags, list) or not 3 <= len(tags) <= 7:
        raise ValueError("tags must be a list of 3–7 tags.")
    normalized_tags = []
    for tag in tags:
        if not isinstance(tag, str):
            raise ValueError("Each tag must be a string.")
        tag = tag.strip().lower()
        if not re.fullmatch(TAG_PATTERN, tag):
            raise ValueError(f"Invalid tag: {tag!r}. Needs lowercase kebab-case.")
        normalized_tags.append(tag)
    data["tags"] = list(dict.fromkeys(normalized_tags))
    if not 3 <= len(data["tags"]) <= 7:
        raise ValueError("Less than 3 tags left after removing duplicates.")
    data["title"] = data["title"].strip()
    data["topic"] = data["topic"].strip()
    if data["subject"] is not None:
        data["subject"] = data["subject"].strip() or None
    return data


def load_or_generate_metadata(transcript, metadata_path, source_fingerprint):
    if metadata_path.exists() and metadata_path.stat().st_size > 0:
        try:
            data = parse_metadata(metadata_path.read_text(encoding="utf-8"))
            print("  Using saved metadata — model call skipped.", flush=True)
            return data
        except Exception as error:
            print(f"  Saved metadata is corrupted ({error}); will be re-fetched.", flush=True)

    previous_error = ""
    previous_response = ""
    for attempt in range(1, MAX_METADATA_ATTEMPTS + 1):
        response = ""
        print(f"  Metadata: attempt {attempt}/{MAX_METADATA_ATTEMPTS}...", flush=True)
        prompt = METADATA_PROMPT.replace("{{TRANSCRIPT}}", transcript)
        if previous_error:
            prompt += METADATA_FIX_HINT.replace("{response}", previous_response[:4000]).replace("{error}", previous_error)
        try:
            schema = METADATA_SCHEMA if RESPONSE_FORMAT_ENABLED else None
            response = model_completion(SYSTEM_PROMPT, prompt, max_tokens=1000, temperature=0.0, json_schema=schema)
            data = parse_metadata(response)
            data_to_save = dict(data)
            if source_fingerprint is not None:
                data_to_save["_source_fingerprint"] = source_fingerprint
            temporary = metadata_path.with_suffix(".json.tmp")
            temporary.write_text(json.dumps(data_to_save, ensure_ascii=False, indent=2), encoding="utf-8")
            temporary.replace(metadata_path)
            print("  Metadata validated and saved.", flush=True)
            return data
        except Exception as error:
            previous_error = str(error)
            previous_response = response
            print(f"  Response failed validation: {previous_error}", flush=True)

    raise RuntimeError(f"Failed to get metadata in {MAX_METADATA_ATTEMPTS} attempts: {previous_error}")


def yaml_scalar(value):
    if value is None:
        return "null"
    return json.dumps(str(value), ensure_ascii=False)


def build_frontmatter(metadata, source_name):
    created = date.today().isoformat()
    lines = [
        "---",
        f"title: {yaml_scalar(metadata['title'])}",
        "tags:",
    ]
    lines.extend(f"  - {yaml_scalar(tag)}" for tag in metadata["tags"])
    lines.extend([
        f"created: {yaml_scalar(created)}",
        f"subject: {yaml_scalar(metadata['subject'])}",
        f"lecture_date: {yaml_scalar(metadata['lecture_date'])}",
        f"source_file: {yaml_scalar(source_name)}",
        "---",
        "",
        f"**{FM_LABELS['date']}:** {metadata['lecture_date'] or FM_LABELS['date_unspecified']}",
        f"**{FM_LABELS['topic']}:** {metadata['topic']}",
        "",
    ])
    return "\n".join(lines)


# Canonical headings and their possible variants found in the model answers.
TRANSLATIONS = {}
for canonical, aliases in SECTION_ALIASES.items():
    for alias in aliases + [canonical]:
        TRANSLATIONS["# " + alias] = "# " + canonical
REQUIRED_HEADINGS = list(SECTION_ALIASES.keys())


def normalize_note_sections(body):
    body = body.strip()
    for source, target in TRANSLATIONS.items():
        body = re.sub(
            r"(?m)^\s*" + re.escape(source) + r"\s*$",
            target,
            body,
        )

    for heading in REQUIRED_HEADINGS:
        if not re.search(r"(?m)^#\s+" + re.escape(heading) + r"\s*$", body):
            fallback = FALLBACK_HOMEWORK if heading == HOMEWORK_NAME else FALLBACK_SECTION
            body += f"\n\n# {heading}\n\n{fallback}"

    return body.strip() + "\n"


def approx_tokens(text):
    return max(1, int(len(text) / 2))


TRANSCRIPT_BUDGET_TOKENS = max(2000, AI_CONTEXT_SIZE - 512 - MAX_NOTE_TOKENS - 1500)


def split_into_chunks(source_text):
    '''Splits a long transcript into parts that fit into a single call.

    Accounting is character-based (one token ~= two characters): a chunk
    accumulated to <= BUDGET * 2 characters is guaranteed to be within the
    token budget with this conservative estimate.
    '''
    budget_chars = TRANSCRIPT_BUDGET_TOKENS * 2
    chunks = []
    current = []
    current_chars = 0

    def flush():
        nonlocal current, current_chars
        if current:
            chunks.append("".join(current))
            current = []
            current_chars = 0

    for line in source_text.splitlines(keepends=True):
        if len(line) > budget_chars:
            flush()
            for start in range(0, len(line), budget_chars):
                piece = line[start:start + budget_chars].strip()
                if piece:
                    chunks.append(piece)
            continue
        if current and current_chars + len(line) > budget_chars:
            flush()
        current.append(line)
        current_chars += len(line)
    flush()
    return [chunk.strip() for chunk in chunks if chunk.strip()]


def generate_note(source_text, metadata):
    chunks = split_into_chunks(source_text)
    part_hint = ""
    if len(chunks) > 1:
        print(
            f"  Transcript is long: about {approx_tokens(source_text)} tokens; processing in {len(chunks)} parts.",
            flush=True,
        )
        part_hint = f"\n\n{PART_NOTE_HINT.format(1, len(chunks))}"

    note_prompt = (
        NOTE_PROMPT_HEADER + "\n\n"
        + METADATA_LABEL + "\n" + json.dumps(metadata, ensure_ascii=False, indent=2) + "\n\n"
        + TEMPLATE_LABEL + "\n" + NOTE_TEMPLATE + "\n\n"
        + TRANSCRIPT_LABEL + "\n" + chunks[0] + part_hint
    )
    note = model_completion(SYSTEM_PROMPT, note_prompt, max_tokens=MAX_NOTE_TOKENS, temperature=0.2)

    for part_index, chunk in enumerate(chunks[1:], 2):
        prompt = (
            INCREMENTAL_PROMPT
            .replace("{{N}}", str(part_index))
            .replace("{{TOTAL}}", str(len(chunks)))
            .replace("{{HEADINGS}}", ", ".join(HEADING_FMT.format(heading) for heading in REQUIRED_HEADINGS))
            .replace("{{DETAILED}}", DETAILED_HEADING)
            .replace("{{NOTE_SO_FAR}}", note.strip() or "(empty)")
            .replace("{{PART}}", chunk)
        )
        note = model_completion(SYSTEM_PROMPT, prompt, max_tokens=MAX_NOTE_TOKENS, temperature=0.2)
        note = strip_reasoning(note).strip()

    return note


def wait_for_server(server, timeout=900):
    started = time.monotonic()
    last_message = started
    while time.monotonic() - started < timeout:
        if server.poll() is not None:
            raise RuntimeError(f"llama-server exited during model loading with code {server.returncode}.")
        try:
            with urllib.request.urlopen(BASE_URL + "/health", timeout=3) as response:
                if response.status == 200:
                    print("llama-server is ready for requests.", flush=True)
                    return
        except Exception:
            pass
        if time.monotonic() - last_message >= 15:
            print("Waiting for Qwen to load into VRAM...", flush=True)
            last_message = time.monotonic()
        time.sleep(2)
    raise TimeoutError("llama-server did not become ready within 15 minutes.")


def audio_fingerprints():
    mapping = {}
    for path in ROOT_DIR.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue
        relative = path.relative_to(ROOT_DIR)
        if any(part in OUTPUT_FOLDER_NAMES for part in relative.parts[:-1]):
            continue
        stem = "__".join(relative.with_suffix("").parts)
        stat = path.stat()
        mapping[stem] = {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
    return mapping


AUDIO_FPS = audio_fingerprints()

# We look for sources by transcripts — a stable correspondence of the audio name to all results.
transcripts = sorted(p for p in TRANSCRIPTS_DIR.glob("*.txt") if p.is_file() and p.stat().st_size > 0)
print(f"Found transcripts: {len(transcripts)}", flush=True)

skip_existing = os.environ.get("SKIP_EXISTING", "1") == "1"
pending = []
for transcript_path in transcripts:
    stem = transcript_path.stem
    note_path = NOTES_DIR / f"{stem}.md"
    if skip_existing and note_path.exists() and note_path.stat().st_size > 0:
        reprocess = False
        current_fp = AUDIO_FPS.get(stem)
        if current_fp is not None and (METADATA_DIR / f"{stem}.json").exists():
            try:
                saved_meta = json.loads((METADATA_DIR / f"{stem}.json").read_text(encoding="utf-8"))
                saved_fp = saved_meta.get("_source_fingerprint")
                if saved_fp is not None and saved_fp != current_fp:
                    reprocess = True
                    print(f"  Audio changed since this note was generated; reprocessing: {note_path.name}", flush=True)
            except Exception:
                pass
        if not reprocess:
            print(f"Skipping — note already exists: {note_path.name}", flush=True)
            continue
    source_path = RAW_DIR / f"{stem}_raw.txt" if AI_INPUT_RAW else transcript_path
    if AI_INPUT_RAW and not source_path.exists():
        print(f"ERROR: Raw input selected, but file is missing: {source_path.name}. Note skipped.", flush=True)
        continue
    if not source_path.exists() or source_path.stat().st_size == 0:
        print(f"ERROR: empty or missing transcript: {source_path}", flush=True)
        continue
    pending.append((transcript_path, source_path, note_path, METADATA_DIR / f"{stem}.json"))

if not pending:
    print("No notes to generate. All files are already processed or there are no suitable sources.", flush=True)
    raise SystemExit(0)

model_path = MODEL_DIR / MODEL_FILENAME
if not model_path.exists() or model_path.stat().st_size < 8_000_000_000:
    print(f"Downloading/checking weights {MODEL_REPO}/{MODEL_FILENAME} to local Colab disk (about 9 GB).", flush=True)
    try:
        downloaded = hf_hub_download(
            repo_id=MODEL_REPO,
            filename=MODEL_FILENAME,
            local_dir=str(MODEL_DIR),
        )
        model_path = Path(downloaded)
    except Exception:
        # If the download was interrupted and created an incomplete file, we delete only the incomplete result.
        if model_path.exists() and model_path.stat().st_size < 8_000_000_000:
            model_path.unlink()
        raise
else:
    print(f"Weights already exist on local Colab disk: {model_path}", flush=True)

server_env = os.environ.copy()
server_env["LD_LIBRARY_PATH"] = LLAMA_LD_LIBRARY_PATH
server_env["CUDA_VISIBLE_DEVICES"] = "0"

base_command = [
    LLAMA_SERVER,
    "--model", str(model_path),
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--alias", "Qwen3-14B-Q4_K_M",
    "--ctx-size", str(AI_CONTEXT_SIZE),
    "--n-gpu-layers", "99",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    # Reuse the KV cache of the shared prompt prefix across requests.
    "--cache-reuse", "256",
    "--batch-size", "256",
    "--ubatch-size", "128",
    "--parallel", "1",
    "--reasoning", "off",
]

server = None
try:
    # If T4 cannot allocate memory for 32K, we try smaller contexts.
    selected_context = None
    for candidate_context in [AI_CONTEXT_SIZE, 24576, 16384]:
        # Do not repeat identical values.
        if selected_context is not None:
            break
        context_command = list(base_command)
        context_command[context_command.index("--ctx-size") + 1] = str(candidate_context)
        print(f"Launching Qwen3-14B Q4_K_M, context {candidate_context}...", flush=True)
        server = subprocess.Popen(context_command, env=server_env)
        try:
            wait_for_server(server, timeout=900)
            selected_context = candidate_context
        except Exception as error:
            if server.poll() is None:
                server.terminate()
                try:
                    server.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    server.kill()
                    server.wait()
            server = None
            if candidate_context == 16384:
                raise
            print(f"Failed to launch context {candidate_context}: {error}", flush=True)
            print("Trying a smaller context size.", flush=True)

    if server is None or selected_context is None:
        raise RuntimeError("Failed to launch Qwen on available VRAM.")
    CONTEXT_SIZE = selected_context

    done = 0
    failed = 0
    from tqdm import tqdm
    for index, (transcript_path, source_path, note_path, metadata_path) in enumerate(
        tqdm(pending, desc="AI notes", unit="file", dynamic_ncols=True, mininterval=1.0), 1
    ):
        print(f"\n[{index}/{len(pending)}] Processing: {source_path.name}", flush=True)
        source_text = source_path.read_text(encoding="utf-8", errors="replace").strip()
        if not source_text:
            failed += 1
            print("ERROR: source text is empty.", flush=True)
            continue

        try:
            stem = transcript_path.stem
            metadata_source = (source_text if approx_tokens(source_text) + 1000 < AI_CONTEXT_SIZE
                               else split_into_chunks(source_text)[0])
            metadata = load_or_generate_metadata(metadata_source, metadata_path, AUDIO_FPS.get(stem))
            print("  Generating full note...", flush=True)
            note_body = generate_note(source_text, metadata)
            note_body = normalize_note_sections(note_body)
            note_text = build_frontmatter(metadata, source_path.name) + note_body
            temporary_note = note_path.with_suffix(".md.tmp")
            temporary_note.write_text(note_text, encoding="utf-8")
            temporary_note.replace(note_path)
            done += 1
            print(f"  Note saved: {note_path}", flush=True)
        except Exception as error:
            failed += 1
            print(f"  ERROR {type(error).__name__}: {error}", flush=True)

    print(f"\nAI stage completed. Successful: {done}, errors: {failed}.", flush=True)
    print(f"Notes: {NOTES_DIR}", flush=True)
finally:
    if server is not None and server.poll() is None:
        print("Stopping llama-server and freeing VRAM...", flush=True)
        server.terminate()
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()
        print("llama-server finished.", flush=True)
"""

    ai_script = (
        ai_template
        .replace("%%SYSTEM_PROMPT%%", PACK["system"])
        .replace("%%METADATA_PROMPT%%", PACK["metadata_prompt"])
        .replace("%%METADATA_FIX_HINT%%", PACK["metadata_fix_hint"])
        .replace("%%NOTE_PROMPT_HEADER%%", PACK["note_header"])
        .replace("%%METADATA_LABEL%%", PACK["metadata_label"])
        .replace("%%TEMPLATE_LABEL%%", PACK["template_label"])
        .replace("%%TRANSCRIPT_LABEL%%", PACK["transcript_label"])
        .replace("%%NOTE_TEMPLATE%%", PACK["note_template"])
        .replace("%%PART_NOTE_HINT%%", PACK["part_note_hint"])
        .replace("%%INCREMENTAL_PROMPT%%", PACK["incremental_prompt"])
        .replace("%%SECTIONS_JSON%%", json.dumps(PACK["sections"], ensure_ascii=False))
        .replace("%%FALLBACK_HOMEWORK%%", PACK["fallback_homework"])
        .replace("%%FALLBACK_SECTION%%", PACK["fallback_section"])
        .replace("%%FM_LABELS_JSON%%", json.dumps(PACK["frontmatter"], ensure_ascii=False))
        .replace("%%FM_DATE%%", PACK["frontmatter"]["date"])
        .replace("%%FM_DATE_UNSPEC%%", PACK["frontmatter"]["date_unspecified"])
        .replace("%%FM_TOPIC%%", PACK["frontmatter"]["topic"])
        .replace("%%DETAILED%%", PACK["detailed_heading"])
        .replace("%%HEADING_FMT_RAW%%", repr(PACK["heading_fmt"]))
    )
    # The Homework section name depends on the note language.
    homework_heading = next(
        (heading for heading in PACK["sections"] if "homework" in heading.lower() or "задание" in heading.lower()),
        "Homework",
    )
    ai_script = ai_script.replace("%%HOMEWORK%%", homework_heading)

    ai_script_path = "/content/lecture_ai_notes.py"
    Path(ai_script_path).write_text(ai_script, encoding="utf-8")

    ai_env = os.environ.copy()
    ai_env.update({
        "INPUT_DIR": str(ROOT_DIR),
        "RAW_DIR": str(RAW_DIR),
        "NOTES_DIR": str(NOTES_DIR),
        "METADATA_DIR": str(METADATA_DIR),
        "MODEL_DIR": str(MODEL_DIR),
        "MODEL_REPO": MODEL_REPO,
        "MODEL_FILENAME": MODEL_FILENAME,
        "LLAMA_SERVER_PATH": str(LLAMA_SERVER_PATH),
        "LLAMA_LD_LIBRARY_PATH": LLAMA_LD_LIBRARY_PATH,
        "MAX_METADATA_ATTEMPTS": str(MAX_METADATA_ATTEMPTS),
        "AI_CONTEXT_SIZE": str(AI_CONTEXT_SIZE),
        "AI_MAX_NOTE_TOKENS": str(AI_MAX_NOTE_TOKENS),
        "SKIP_EXISTING": "1" if SKIP_EXISTING else "0",
        "PYTHONUNBUFFERED": "1",
    })

    print("Launching a separate Qwen process. The transcription process is already finished, so models don't share VRAM.")
    ai_env["AI_INPUT_RAW"] = "1" if AI_INPUT_RAW else "0"
    run_streaming([PYTHON, "-u", ai_script_path], ai_env)
    print("AI notes stage completed. Qwen stopped, VRAM freed.")
